In [ ]:
# ============================================
# Student Dropout Risk Analysis System
# Frontend: Streamlit | Backend: Python + ML
# ============================================

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)


In [ ]:
st.set_page_config(
    page_title="Student Dropout Risk Prediction",
    layout="wide"
)

st.title("🎓 Student Retention & Dropout Risk Prediction System")
st.write("Machine Learning–based analysis to identify students at risk of dropping out")


In [1]:
@st.cache_data
def load_data():
    return pd.read_csv("student_dropout.csv")

df = load_data()

st.subheader("📊 Dataset Overview")
st.write("Dataset Shape:", df.shape)
st.dataframe(df.head())


NameError: name 'st' is not defined

In [ ]:
st.subheader("📈 Dropout Distribution")

fig, ax = plt.subplots()
sns.countplot(x="next_semester_dropout", data=df, ax=ax)
ax.set_title("Dropout Distribution")
st.pyplot(fig)


In [ ]:
X = df.drop("next_semester_dropout", axis=1)
y = df["next_semester_dropout"]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns


In [ ]:
numeric_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [ ]:
st.sidebar.header("⚙️ Model Selection")
model_choice = st.sidebar.selectbox(
    "Choose ML Model",
    ["Logistic Regression", "Random Forest"]
)


In [ ]:
if model_choice == "Logistic Regression":
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000
        ))
    ])
else:
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42
        ))
    ])

model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

st.subheader("📌 Model Performance")

st.text("Classification Report")
st.text(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_prob)
st.write("ROC AUC Score:", roc_auc)


In [ ]:
fig_cm, ax_cm = plt.subplots()
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax_cm)
ax_cm.set_title("Confusion Matrix")
ax_cm.set_xlabel("Predicted")
ax_cm.set_ylabel("Actual")
st.pyplot(fig_cm)


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig_roc, ax_roc = plt.subplots()
ax_roc.plot(fpr, tpr, label="ROC Curve")
ax_roc.plot([0, 1], [0, 1], "--")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.set_title("ROC Curve")
ax_roc.legend()
st.pyplot(fig_roc)


In [ ]:
if model_choice == "Random Forest":
    st.subheader("🔍 Feature Importance")

    rf_clf = model.named_steps["classifier"]
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": rf_clf.feature_importances_
    }).sort_values(by="Importance", ascending=False)

    st.dataframe(importance_df.head(10))


In [ ]:
st.subheader("⚠️ Dropout Risk Score Distribution")

fig_risk, ax_risk = plt.subplots()
sns.histplot(y_prob, bins=30, kde=True, ax=ax_risk)
ax_risk.set_title("Dropout Risk Scores")
ax_risk.set_xlabel("Probability of Dropout")
st.pyplot(fig_risk)
